# EDA — Plant 1 Generation Data

Exploration of `Plant_1_Generation_Data.csv` to understand the structure, distributions, and patterns before modeling.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns

plt.rcParams["figure.figsize"] = (14, 5)
sns.set_theme(style="whitegrid")

DATA_PATH = "../data/raw/Plant_1_Generation_Data.csv"

## 1. Chargement et aperçu général

In [ ]:
df = pd.read_csv(DATA_PATH, parse_dates=["DATE_TIME"], dayfirst=True)
print(f"Shape : {df.shape}")
print(f"Période : {df['DATE_TIME'].min()} → {df['DATE_TIME'].max()}")
print(f"Onduleurs (SOURCE_KEY) : {df['SOURCE_KEY'].nunique()}")
df.head()

In [ ]:
df.dtypes

In [ ]:
df.describe()

## 2. Valeurs manquantes

In [ ]:
missing = df.isnull().sum()
print(missing[missing > 0] if missing.any() else "Aucune valeur manquante.")

# Vérification doublons
dupes = df.duplicated(subset=["DATE_TIME", "SOURCE_KEY"]).sum()
print(f"\nDoublons (DATE_TIME + SOURCE_KEY) : {dupes}")

## 3. Distribution des variables numériques

In [ ]:
num_cols = ["DC_POWER", "AC_POWER", "DAILY_YIELD", "TOTAL_YIELD"]

# Filtrage heures de jour (DC_POWER > 0) pour éviter le biais de la nuit
df_day = df[df["DC_POWER"] > 0]

fig, axes = plt.subplots(1, 4, figsize=(18, 4))
for ax, col in zip(axes, num_cols):
    df_day[col].hist(bins=60, ax=ax, color="steelblue", edgecolor="none")
    ax.set_title(col)
    ax.set_xlabel("")
plt.suptitle("Distributions (heures de jour uniquement)", y=1.02)
plt.tight_layout()
plt.show()

## 4. Série temporelle — production globale

In [ ]:
total_by_time = df.groupby("DATE_TIME")[["DC_POWER", "AC_POWER"]].sum()

fig, axes = plt.subplots(2, 1, figsize=(16, 7), sharex=True)
axes[0].plot(total_by_time.index, total_by_time["DC_POWER"], linewidth=0.6, color="darkorange")
axes[0].set_ylabel("DC_POWER (W)")
axes[0].set_title("Production DC totale (tous onduleurs)")

axes[1].plot(total_by_time.index, total_by_time["AC_POWER"], linewidth=0.6, color="steelblue")
axes[1].set_ylabel("AC_POWER (W)")
axes[1].set_title("Production AC totale")
axes[1].xaxis.set_major_formatter(mdates.DateFormatter("%d-%b"))

plt.tight_layout()
plt.show()

## 5. Profil journalier moyen (par heure)

In [ ]:
df["hour"] = df["DATE_TIME"].dt.hour
hourly = df.groupby("hour")[["DC_POWER", "AC_POWER"]].mean()

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(hourly.index, hourly["DC_POWER"], label="DC_POWER", color="darkorange", marker="o", markersize=4)
ax.plot(hourly.index, hourly["AC_POWER"], label="AC_POWER", color="steelblue", marker="o", markersize=4)
ax.set_xlabel("Heure")
ax.set_ylabel("Puissance moyenne (W)")
ax.set_title("Profil journalier moyen (tous onduleurs, tous jours)")
ax.legend()
plt.tight_layout()
plt.show()

## 6. Production par onduleur (SOURCE_KEY)

In [ ]:
inv_stats = (
    df_day.groupby("SOURCE_KEY")["DC_POWER"]
    .agg(mean="mean", std="std", median="median")
    .sort_values("mean", ascending=False)
)
print(inv_stats.to_string())

fig, ax = plt.subplots(figsize=(16, 5))
inv_stats["mean"].plot(kind="bar", ax=ax, color="steelblue", yerr=inv_stats["std"], capsize=3)
ax.set_title("DC_POWER moyen par onduleur (± std) — heures de jour")
ax.set_xlabel("SOURCE_KEY")
ax.set_ylabel("DC_POWER (W)")
ax.tick_params(axis="x", rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
# Heatmap : DC_POWER moyen par onduleur × heure
pivot = df_day.assign(hour=df_day["DATE_TIME"].dt.hour).pivot_table(
    values="DC_POWER", index="SOURCE_KEY", columns="hour", aggfunc="mean"
)

fig, ax = plt.subplots(figsize=(16, 9))
sns.heatmap(pivot, cmap="YlOrRd", ax=ax, linewidths=0.3, fmt=".0f", annot=False)
ax.set_title("DC_POWER moyen par onduleur et par heure")
ax.set_xlabel("Heure")
ax.set_ylabel("Onduleur (SOURCE_KEY)")
plt.tight_layout()
plt.show()

## 7. Relation DC_POWER / AC_POWER (rendement onduleur)

In [ ]:
df_day = df_day.copy()
df_day["efficiency"] = df_day["AC_POWER"] / df_day["DC_POWER"]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Scatter DC vs AC
axes[0].scatter(df_day["DC_POWER"], df_day["AC_POWER"], alpha=0.05, s=2, color="steelblue")
axes[0].set_xlabel("DC_POWER (W)")
axes[0].set_ylabel("AC_POWER (W)")
axes[0].set_title("DC vs AC Power")

# Distribution du rendement
df_day["efficiency"].clip(0, 1.2).hist(bins=80, ax=axes[1], color="darkorange", edgecolor="none")
axes[1].axvline(1.0, color="red", linestyle="--", label="100 %")
axes[1].set_xlabel("Rendement AC/DC")
axes[1].set_title("Distribution du rendement onduleur")
axes[1].legend()

plt.tight_layout()
plt.show()

print(f"Rendement médian : {df_day['efficiency'].median():.3f}")
print(f"Rendement < 0.9  : {(df_day['efficiency'] < 0.9).mean():.1%} des mesures de jour")

## 8. Production journalière totale (DAILY_YIELD)

In [ ]:
df["date"] = df["DATE_TIME"].dt.date

# DAILY_YIELD en fin de journée = max par onduleur × jour
daily_max = df.groupby(["date", "SOURCE_KEY"])["DAILY_YIELD"].max().reset_index()
daily_total = daily_max.groupby("date")["DAILY_YIELD"].sum()

fig, ax = plt.subplots(figsize=(14, 4))
ax.bar(range(len(daily_total)), daily_total.values, color="steelblue", width=0.8)
ax.set_xticks(range(len(daily_total)))
ax.set_xticklabels([str(d) for d in daily_total.index], rotation=45, ha="right")
ax.set_ylabel("Production journalière totale (kWh)")
ax.set_title("Production journalière cumulée — tous onduleurs")
plt.tight_layout()
plt.show()

## 9. Boxplots DC_POWER par onduleur — détection visuelle d'anomalies

In [ ]:
order = inv_stats.sort_values("mean", ascending=False).index

fig, ax = plt.subplots(figsize=(18, 6))
df_day.boxplot(
    column="DC_POWER",
    by="SOURCE_KEY",
    ax=ax,
    notch=False,
    sym=".",
    flierprops=dict(marker=".", markersize=2, alpha=0.3),
)
ax.set_title("DC_POWER par onduleur — heures de jour")
ax.set_xlabel("SOURCE_KEY")
ax.set_ylabel("DC_POWER (W)")
plt.suptitle("")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## 10. Corrélations

In [ ]:
corr = df_day[["DC_POWER", "AC_POWER", "DAILY_YIELD", "efficiency"]].corr()

fig, ax = plt.subplots(figsize=(7, 5))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", ax=ax, vmin=-1, vmax=1)
ax.set_title("Matrice de corrélation — heures de jour")
plt.tight_layout()
plt.show()

## 11. Observations & pistes pour la modélisation

- **Colonnes clés** : `DC_POWER`, `AC_POWER`, `DAILY_YIELD` ; `TOTAL_YIELD` est cumulatif et peu utile pour la détection.
- **Filtrage nuit** : appliquer `DC_POWER > 0` (ou `IRRADIATION > 0.01` une fois le fichier météo joint) avant de modéliser.
- **Feature `perf_ratio`** : nécessite de joindre `Plant_1_Weather_Sensor_Data.csv` pour avoir `IRRADIATION`.
- **Variabilité inter-onduleurs** : certains onduleurs montrent une distribution décalée vers le bas — candidats naturels à l'anomalie.
- **Rendement AC/DC** : médiane ~0.97 ; valeurs < 0.9 méritent investigation (pertes anormales).
- **Contamination Isolation Forest** : commencer à 0.05, ajuster après validation visuelle sur les boxplots.

---
# Isolation Forest — Détection d'anomalies

## 12. Feature Engineering

In [ ]:
WEATHER_PATH = "../data/raw/Plant_1_Weather_Sensor_Data.csv"

weather = pd.read_csv(WEATHER_PATH, parse_dates=["DATE_TIME"])
# Le fichier météo n'a qu'un capteur : on garde uniquement DATE_TIME + mesures
weather = weather[["DATE_TIME", "IRRADIATION", "AMBIENT_TEMPERATURE", "MODULE_TEMPERATURE"]].copy()

# Harmonisation du format datetime (génération = dayfirst, météo = isoformat)
df["DATE_TIME"] = pd.to_datetime(df["DATE_TIME"])
df_feat = df.merge(weather, on="DATE_TIME", how="left")

# Filtre nuit (IRRADIATION > 0.01 kW/m²)
df_feat = df_feat[df_feat["IRRADIATION"] > 0.01].copy()

# Feature : rendement DC/AC
df_feat["efficiency"] = df_feat["AC_POWER"] / df_feat["DC_POWER"]

# Feature : performance ratio = puissance DC normalisée par l'irradiation
df_feat["perf_ratio"] = df_feat["DC_POWER"] / df_feat["IRRADIATION"]

# Feature : écart à la médiane des onduleurs pour ce même horodatage
median_by_ts = df_feat.groupby("DATE_TIME")["DC_POWER"].transform("median")
df_feat["power_deviation"] = df_feat["DC_POWER"] - median_by_ts

print(f"Lignes après filtre nuit : {len(df_feat):,}")
print(f"Features créées : efficiency, perf_ratio, power_deviation")
df_feat[["DATE_TIME", "SOURCE_KEY", "DC_POWER", "IRRADIATION",
         "perf_ratio", "power_deviation", "efficiency"]].head()

In [ ]:
# Distribution des 3 nouvelles features
fig, axes = plt.subplots(1, 3, figsize=(17, 4))

df_feat["perf_ratio"].clip(0, df_feat["perf_ratio"].quantile(0.99)).hist(
    bins=80, ax=axes[0], color="steelblue", edgecolor="none")
axes[0].set_title("perf_ratio (DC / Irradiation)")

df_feat["power_deviation"].hist(
    bins=80, ax=axes[1], color="darkorange", edgecolor="none")
axes[1].axvline(0, color="red", linestyle="--")
axes[1].set_title("power_deviation (écart à la médiane)")

df_feat["efficiency"].clip(0, 1.1).hist(
    bins=80, ax=axes[2], color="seagreen", edgecolor="none")
axes[2].set_title("efficiency (AC/DC)")

plt.suptitle("Distribution des features d'entrée (heures de jour)", y=1.02)
plt.tight_layout()
plt.show()

## 13. Entraînement de l'Isolation Forest

In [ ]:
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler

FEATURES = ["DC_POWER", "perf_ratio", "power_deviation", "efficiency"]
CONTAMINATION = 0.05

X = df_feat[FEATURES].dropna()
idx = X.index  # pour réaligner après scaling

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)  # scaler fitté uniquement sur les données de jour

model = IsolationForest(contamination=CONTAMINATION, n_estimators=200, random_state=42, n_jobs=-1)
model.fit(X_scaled)

df_feat.loc[idx, "anomaly_score"] = -model.score_samples(X_scaled)   # score : + élevé = + anormal
df_feat.loc[idx, "anomaly"]       = model.predict(X_scaled)          # -1 = anomalie, +1 = normal

n_anomalies = (df_feat["anomaly"] == -1).sum()
print(f"Anomalies détectées : {n_anomalies:,} / {len(df_feat):,} ({n_anomalies/len(df_feat):.1%})")

## 14. Distribution des scores d'anomalie

In [ ]:
threshold = df_feat.loc[df_feat["anomaly"] == -1, "anomaly_score"].min()

fig, ax = plt.subplots(figsize=(12, 4))
df_feat[df_feat["anomaly"] == 1]["anomaly_score"].hist(
    bins=100, ax=ax, color="steelblue", alpha=0.7, label="Normal", density=True)
df_feat[df_feat["anomaly"] == -1]["anomaly_score"].hist(
    bins=100, ax=ax, color="crimson", alpha=0.7, label="Anomalie", density=True)
ax.axvline(threshold, color="black", linestyle="--", linewidth=1.2, label=f"Seuil ({threshold:.3f})")
ax.set_xlabel("Anomaly Score (plus élevé = plus anormal)")
ax.set_ylabel("Densité")
ax.set_title("Distribution des scores d'anomalie — Isolation Forest")
ax.legend()
plt.tight_layout()
plt.show()

## 15. Scatter plots — anomalies dans l'espace des features

In [ ]:
normal = df_feat[df_feat["anomaly"] == 1]
anomaly = df_feat[df_feat["anomaly"] == -1]

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

pairs = [
    ("DC_POWER", "perf_ratio"),
    ("DC_POWER", "power_deviation"),
    ("perf_ratio", "efficiency"),
]
labels = [
    ("DC_POWER (W)", "Perf Ratio (DC/Irrad)"),
    ("DC_POWER (W)", "Power Deviation (W)"),
    ("Perf Ratio", "Efficiency (AC/DC)"),
]

for ax, (x, y), (xl, yl) in zip(axes, pairs, labels):
    ax.scatter(normal[x], normal[y], s=2, alpha=0.05, color="steelblue", label="Normal")
    ax.scatter(anomaly[x], anomaly[y], s=6, alpha=0.4, color="crimson", label="Anomalie")
    ax.set_xlabel(xl)
    ax.set_ylabel(yl)
    ax.set_title(f"{x} vs {y}")

axes[0].legend(markerscale=4)
plt.suptitle("Anomalies dans l'espace des features", fontsize=13)
plt.tight_layout()
plt.show()

## 16. Taux d'anomalie par onduleur

In [ ]:
anomaly_rate = (
    df_feat.groupby("SOURCE_KEY")["anomaly"]
    .apply(lambda s: (s == -1).mean())
    .sort_values(ascending=False)
    .rename("anomaly_rate")
)

colors = ["crimson" if r > CONTAMINATION * 1.5 else "steelblue" for r in anomaly_rate]

fig, ax = plt.subplots(figsize=(16, 5))
anomaly_rate.plot(kind="bar", ax=ax, color=colors)
ax.axhline(CONTAMINATION, color="black", linestyle="--", linewidth=1.2, label=f"contamination = {CONTAMINATION}")
ax.set_title("Taux d'anomalie par onduleur (contamination = 5 %)")
ax.set_xlabel("SOURCE_KEY")
ax.set_ylabel("Taux d'anomalie")
ax.set_ylim(0, anomaly_rate.max() * 1.2)
ax.tick_params(axis="x", rotation=45)
ax.legend()
plt.tight_layout()
plt.show()

print("Top 5 onduleurs les plus anormaux :")
print(anomaly_rate.head(5).to_string())

## 17. Heatmap anomalies — onduleur × jour

In [ ]:
df_feat["date"] = df_feat["DATE_TIME"].dt.date

heatmap_data = (
    df_feat.groupby(["SOURCE_KEY", "date"])["anomaly"]
    .apply(lambda s: (s == -1).mean())
    .unstack(level="date")
)

# Trier les onduleurs par taux d'anomalie global (décroissant)
heatmap_data = heatmap_data.loc[anomaly_rate.index]

fig, ax = plt.subplots(figsize=(18, 10))
sns.heatmap(
    heatmap_data,
    cmap="Reds",
    ax=ax,
    linewidths=0.3,
    cbar_kws={"label": "Taux d'anomalie"},
    vmin=0, vmax=1,
)
ax.set_title("Taux d'anomalie par onduleur et par jour")
ax.set_xlabel("Date")
ax.set_ylabel("Onduleur (SOURCE_KEY)")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

## 18. Série temporelle — score d'anomalie agrégé

In [ ]:
ts_score = df_feat.groupby("DATE_TIME")["anomaly_score"].mean()
ts_count = df_feat.groupby("DATE_TIME").apply(lambda g: (g["anomaly"] == -1).sum())

fig, axes = plt.subplots(2, 1, figsize=(16, 7), sharex=True)

axes[0].plot(ts_score.index, ts_score.values, linewidth=0.7, color="darkorange")
axes[0].set_ylabel("Score moyen")
axes[0].set_title("Score d'anomalie moyen (tous onduleurs)")

axes[1].fill_between(ts_count.index, ts_count.values, alpha=0.6, color="crimson")
axes[1].set_ylabel("Nb anomalies")
axes[1].set_title("Nombre d'onduleurs en anomalie à chaque instant")
axes[1].xaxis.set_major_formatter(mdates.DateFormatter("%d-%b"))

plt.tight_layout()
plt.show()

## 19. Deep dive — top 3 onduleurs anormaux

In [ ]:
top3 = anomaly_rate.head(3).index.tolist()

fig, axes = plt.subplots(3, 1, figsize=(16, 12), sharex=True)

for ax, key in zip(axes, top3):
    sub = df_feat[df_feat["SOURCE_KEY"] == key].sort_values("DATE_TIME")
    norm = sub[sub["anomaly"] == 1]
    anom = sub[sub["anomaly"] == -1]

    ax.scatter(norm["DATE_TIME"], norm["DC_POWER"], s=4, alpha=0.4,
               color="steelblue", label="Normal")
    ax.scatter(anom["DATE_TIME"], anom["DC_POWER"], s=18, alpha=0.8,
               color="crimson", marker="x", label="Anomalie")
    ax.set_ylabel("DC_POWER (W)")
    ax.set_title(f"Onduleur : {key}  (taux anomalie : {anomaly_rate[key]:.1%})")
    ax.legend(loc="upper right", markerscale=2)

axes[-1].xaxis.set_major_formatter(mdates.DateFormatter("%d-%b"))
plt.suptitle("DC_POWER avec anomalies — Top 3 onduleurs", fontsize=13)
plt.tight_layout()
plt.show()

## 20. Profil moyen : onduleurs anomaux vs normaux

In [ ]:
# Classer chaque onduleur comme "anormal" si son taux > 1.5× contamination
high_anomaly_keys = anomaly_rate[anomaly_rate > CONTAMINATION * 1.5].index
low_anomaly_keys  = anomaly_rate[anomaly_rate <= CONTAMINATION * 1.5].index

hourly_high = (
    df_feat[df_feat["SOURCE_KEY"].isin(high_anomaly_keys)]
    .groupby("hour")["DC_POWER"].mean()
)
hourly_low = (
    df_feat[df_feat["SOURCE_KEY"].isin(low_anomaly_keys)]
    .groupby("hour")["DC_POWER"].mean()
)

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(hourly_low.index, hourly_low.values, color="steelblue", marker="o",
        markersize=4, label=f"Normaux ({len(low_anomaly_keys)} onduleurs)")
ax.plot(hourly_high.index, hourly_high.values, color="crimson", marker="o",
        markersize=4, label=f"Anormaux ({len(high_anomaly_keys)} onduleurs)")
ax.fill_between(hourly_low.index, hourly_high.values, hourly_low.values,
                alpha=0.15, color="crimson", label="Écart de production")
ax.set_xlabel("Heure")
ax.set_ylabel("DC_POWER moyen (W)")
ax.set_title("Profil journalier moyen : onduleurs normaux vs anormaux")
ax.legend()
plt.tight_layout()
plt.show()

gap = (hourly_low - hourly_high).mean()
print(f"Perte moyenne estimée (heures actives) : {gap:.0f} W / onduleur")

## 21. Sensibilité à la contamination

In [ ]:
contam_values = [0.02, 0.05, 0.08, 0.12, 0.20]
results = {}

for c in contam_values:
    m = IsolationForest(contamination=c, n_estimators=200, random_state=42, n_jobs=-1)
    preds = m.fit_predict(X_scaled)
    rate_per_inv = pd.Series(preds, index=idx).rename("p")
    rate_per_inv = df_feat.loc[idx, ["SOURCE_KEY"]].join(rate_per_inv)
    results[c] = (
        rate_per_inv.groupby("SOURCE_KEY")["p"]
        .apply(lambda s: (s == -1).mean())
        .sort_index()
    )

rates_df = pd.DataFrame(results)

fig, ax = plt.subplots(figsize=(17, 5))
x = np.arange(len(rates_df))
width = 0.15
offsets = np.linspace(-2, 2, len(contam_values)) * width

cmap = plt.cm.get_cmap("plasma", len(contam_values))
for i, (c, offset) in enumerate(zip(contam_values, offsets)):
    ax.bar(x + offset, rates_df[c].values, width=width, color=cmap(i),
           alpha=0.85, label=f"c={c}")

ax.set_xticks(x)
ax.set_xticklabels(rates_df.index, rotation=45, ha="right")
ax.set_ylabel("Taux d'anomalie")
ax.set_title("Taux d'anomalie par onduleur selon la contamination")
ax.legend(title="contamination", bbox_to_anchor=(1.01, 1), loc="upper left")
plt.tight_layout()
plt.show()

## 22. Importance des features (via score moyen des arbres)

In [ ]:
# Profondeur moyenne d'isolation pour chaque feature (proxy d'importance)
# On perturbe une feature à la fois et mesure le changement de score moyen
baseline_score = -model.score_samples(X_scaled).mean()
importances = {}

rng = np.random.default_rng(42)
for i, feat in enumerate(FEATURES):
    X_perm = X_scaled.copy()
    X_perm[:, i] = rng.permutation(X_perm[:, i])
    perm_score = -model.score_samples(X_perm).mean()
    importances[feat] = perm_score - baseline_score

imp_series = pd.Series(importances).sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(8, 4))
imp_series.plot(kind="barh", ax=ax, color=["crimson" if v > 0 else "steelblue" for v in imp_series])
ax.axvline(0, color="black", linewidth=0.8)
ax.set_title("Importance des features (permutation sur anomaly score)")
ax.set_xlabel("Δ score moyen (+ = feature contribue à la détection)")
plt.tight_layout()
plt.show()

## 23. Score d'anomalie vs Irradiation (contexte météo)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sc = axes[0].scatter(
    df_feat["IRRADIATION"], df_feat["anomaly_score"],
    c=df_feat["anomaly_score"], cmap="RdYlBu_r",
    s=3, alpha=0.3, vmin=df_feat["anomaly_score"].quantile(0.01),
    vmax=df_feat["anomaly_score"].quantile(0.99),
)
plt.colorbar(sc, ax=axes[0], label="Anomaly Score")
axes[0].set_xlabel("Irradiation (kW/m²)")
axes[0].set_ylabel("Anomaly Score")
axes[0].set_title("Score vs Irradiation")

sc2 = axes[1].scatter(
    df_feat["MODULE_TEMPERATURE"], df_feat["anomaly_score"],
    c=df_feat["anomaly_score"], cmap="RdYlBu_r",
    s=3, alpha=0.3, vmin=df_feat["anomaly_score"].quantile(0.01),
    vmax=df_feat["anomaly_score"].quantile(0.99),
)
plt.colorbar(sc2, ax=axes[1], label="Anomaly Score")
axes[1].set_xlabel("Module Temperature (°C)")
axes[1].set_ylabel("Anomaly Score")
axes[1].set_title("Score vs Température module")

plt.suptitle("Anomaly Score en fonction du contexte météo", fontsize=13)
plt.tight_layout()
plt.show()

## 24. Tableau récapitulatif final

In [ ]:
summary = (
    df_feat.groupby("SOURCE_KEY")
    .agg(
        dc_mean=("DC_POWER", "mean"),
        dc_median=("DC_POWER", "median"),
        perf_ratio_mean=("perf_ratio", "mean"),
        efficiency_mean=("efficiency", "mean"),
        anomaly_score_mean=("anomaly_score", "mean"),
        anomaly_rate=("anomaly", lambda s: (s == -1).mean()),
        n_obs=("DC_POWER", "count"),
    )
    .sort_values("anomaly_rate", ascending=False)
    .round(3)
)

# Coloration conditionnelle pour lecture rapide
summary.style.background_gradient(subset=["anomaly_rate"], cmap="Reds") \
             .background_gradient(subset=["dc_mean"], cmap="Blues") \
             .format({"anomaly_rate": "{:.1%}", "dc_mean": "{:.0f}", "dc_median": "{:.0f}"})